# 01. 셀 단위 표현과 2방향 attention

TabPFN v2(Hollmann et al., *Nature* 2025)는 v1과 같은 PFN 학습 방식을 쓰지만, Transformer에 넣는 토큰의 단위가 다릅니다. v1은 표의 **행** 하나가 토큰 하나였고, v2는 표의 **셀** 하나가 토큰 하나입니다. 그리고 한 층 안에서 같은 행의 셀끼리(feature attention), 같은 열의 셀끼리(sample attention) 번갈아 attend합니다(논문 Fig. 1b, Methods의 "Details on the neural architecture").

이 노트북은 논문을 읽기 전에 이 구조를 작은 표로 따라갑니다. 이해용 작은 모델은 노트북 안에서 직접 만듭니다. 학습에 쓰는 실제 모델은 같은 구조를 키워 `src/minipfn/tabpfn_v2/model.py`에 두었습니다.

다루는 흐름:

1. v1: 행 하나가 토큰 하나
2. v2: 셀 하나가 토큰 하나
3. feature attention — 같은 행의 셀끼리
4. sample attention — 같은 열의 셀끼리
5. 한 층 쌓기와 작은 모델
6. 구조가 보장하는 성질
7. 열에 이름표 붙이기 — random feature embedding
8. 셀 2개를 토큰 하나로 — feature group
9. 학습 행은 한 번만 계산하기 — 상태 캐시
10. 계산량
11. 정리

## `minipfn` 불러오기

다른 노트북과 같은 import shim입니다. 이 노트북은 `notebooks/03_tabpfn_v2/` 아래에 있으므로 저장소 루트는 두 단계 위입니다.

In [1]:
try:
    import minipfn
except ImportError:
    import sys
    from pathlib import Path

    sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
    import minipfn

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn

from minipfn import set_seed

set_seed(0)  # 재현성을 위해 난수 시드를 고정한다.

## 1. v1: 행 하나가 토큰 하나

v1 walkthrough와 같은 과일 가게 표를 씁니다. 무게와 당도로 사과(0)와 배(1)를 구분하고, 정답을 아는 학습 행 4개와 정답을 가린 테스트 행 2개가 있습니다.

In [3]:
fruit = pd.DataFrame(
    {
        "무게(g)": [210, 190, 320, 290, 200, 310],
        "당도(brix)": [13.5, 14.0, 11.0, 11.5, 13.0, 12.0],
        "y": [0, 0, 1, 1, np.nan, np.nan],  # 테스트 행의 정답은 모른다 (NaN)
    }
)
n_train = 4
roles = ["학습"] * n_train + ["테스트"] * 2

shown = fruit.copy()
shown["y"] = ["0 (사과)", "0 (사과)", "1 (배)", "1 (배)", "?", "?"]
shown.insert(0, "역할", roles)
display(shown)

,역할,무게(g),당도(brix),y
0,학습,210,13.5,0 (사과)
1,학습,190,14.0,0 (사과)
2,학습,320,11.0,1 (배)
3,학습,290,11.5,1 (배)
4,테스트,200,13.0,?
5,테스트,310,12.0,?


v1은 행 하나의 feature 벡터 전체를 Linear 하나(`x_encoder`, feature 100개 → 512)에 넣어 토큰 하나를 만들었습니다. 이 Linear는 열 위치마다 다른 가중치를 쓰므로 두 가지 제약이 생깁니다.

- 열 순서를 바꾸면 같은 행이라도 토큰이 달라집니다.
- 입력 크기가 100으로 고정되어, feature가 100개를 넘는 표는 넣을 수 없습니다. 적은 표는 0으로 채웁니다.

첫 학습 행으로 첫째 제약을 확인합니다. 값은 학습 행 기준으로 표준화한 값입니다(2절에서 다시 설명합니다).

In [4]:
features = fruit[["무게(g)", "당도(brix)"]].to_numpy(dtype=float)
z = (features - features[:n_train].mean(axis=0)) / features[:n_train].std(axis=0)

torch.manual_seed(0)
v1_encoder = nn.Linear(2, 4)  # v1의 x_encoder를 줄인 것: feature 2개 → 임베딩 4

row = torch.tensor(z[0], dtype=torch.float32)  # 첫 학습 행 (무게, 당도)
swapped = row.flip(0)  # 같은 행의 열 순서만 바꾼 것 (당도, 무게)
with torch.no_grad():
    display(
        pd.DataFrame(
            {"원래 순서": v1_encoder(row).numpy(), "열 순서 바꿈": v1_encoder(swapped).numpy()},
            index=[f"임베딩 {i}" for i in range(4)],
        ).round(3)
    )

,원래 순서,열 순서 바꿈
임베딩 0,0.239,-0.365
임베딩 1,0.237,0.140
임베딩 2,0.149,-0.577
임베딩 3,0.312,-0.591


같은 정보인데 토큰이 다릅니다. 그래서 v1은 학습과 추론 때 열 순서를 무작위로 돌려, 모델이 열 순서에 덜 민감해지도록 배우게 했습니다.

## 2. v2: 셀 하나가 토큰 하나

v2는 셀 하나를 토큰 하나로 만듭니다. 셀 값 하나는 세 단계를 거쳐 토큰이 됩니다. 공식 구현(`TabPFNV2._embed_features`, `_embed_targets`)의 순서를 줄인 것입니다.

**1단계: 열마다 표준화합니다.** 평균과 표준편차는 학습 행에서만 구하고, 테스트 행에도 같은 값을 씁니다.

In [5]:
# 1단계: 열마다 학습 행의 평균·표준편차로 표준화한다
mean = features[:n_train].mean(axis=0)
std = features[:n_train].std(axis=0)
z = (features - mean) / std

display(pd.DataFrame(z, columns=["무게 (표준화)", "당도 (표준화)"], index=roles).round(2))

,무게 (표준화),당도 (표준화)
학습,-0.79,0.78
학습,-1.16,1.18
학습,1.25,-1.18
학습,0.69,-0.78
테스트,-0.97,0.39
테스트,1.06,-0.39


**2단계: 빈 칸을 채우고 표시를 붙입니다.** 테스트 행의 y는 모르는 값이라 비어 있습니다. 빈 칸은 학습 행의 평균으로 채우고, "비어 있었다"는 표시 값을 따로 붙입니다. 공식 구현은 빈 칸이면 $-2$, 아니면 $0$을 붙입니다. feature에 결측값이 있을 때도 같은 방식입니다. 그래서 v2는 결측값을 따로 전처리하지 않아도 됩니다.

In [6]:
NAN_INDICATOR = -2.0  # 공식 구현이 빈 칸에 붙이는 표시 값

y = fruit["y"].to_numpy(dtype=float)
y_missing = np.isnan(y)  # 테스트 행의 y는 비어 있다
y_filled = np.where(y_missing, y[:n_train].mean(), y)  # 학습 행 평균(0.5)으로 채운다
y_indicator = np.where(y_missing, NAN_INDICATOR, 0.0)  # 비어 있었으면 -2

display(pd.DataFrame({"y 원래 값": y, "채운 값": y_filled, "빈 칸 표시": y_indicator}, index=roles))

,y 원래 값,채운 값,빈 칸 표시
학습,0.0,0.0,0.0
학습,0.0,0.0,0.0
학습,1.0,1.0,0.0
학습,1.0,1.0,0.0
테스트,NaN,0.5,-2.0
테스트,NaN,0.5,-2.0


**3단계: [값, 빈 칸 표시] 두 숫자를 Linear로 토큰으로 바꿉니다.** feature 열은 모두 같은 Linear(`feature_embedder`)를 함께 쓰고, y 열은 전용 Linear(`target_embedder`)를 씁니다. 어떤 열이든 입력 크기가 2이므로 feature 수에 제한이 없습니다.

In [7]:
E = 4  # 임베딩 크기 (공식 v2는 192)

torch.manual_seed(0)
feature_embedder = nn.Linear(2, E, bias=False)  # 모든 feature 열이 공유
target_embedder = nn.Linear(2, E)  # y 열 전용

# feature 셀: [표준화한 값, 빈 칸 표시]. 이 표에는 feature 결측값이 없으므로 표시는 모두 0
x_cells = torch.stack(
    [torch.tensor(z, dtype=torch.float32), torch.zeros(6, 2)], dim=-1
)  # (행 6, 열 2, 2)
y_cells = torch.tensor(np.stack([y_filled, y_indicator], axis=-1), dtype=torch.float32)
y_cells = y_cells.unsqueeze(1)  # (행 6, 열 1, 2)

with torch.no_grad():
    x_tokens = feature_embedder(x_cells)  # (행 6, 열 2, E)
    y_tokens = target_embedder(y_cells)  # (행 6, 열 1, E)
tokens = torch.cat([x_tokens, y_tokens], dim=1)  # y 토큰을 마지막 열로 붙인다

print("토큰 텐서:", tuple(tokens.shape), " (행, 열, 임베딩)")
print("v1 토큰 수:", len(fruit), " (행마다 1개)")
print("v2 토큰 수:", tokens.shape[0] * tokens.shape[1], " (셀마다 1개)")

토큰 텐서: (6, 3, 4)  (행, 열, 임베딩)
v1 토큰 수: 6  (행마다 1개)
v2 토큰 수: 18  (셀마다 1개)


토큰은 행 × 열 격자로 놓입니다. 아래 표의 칸 하나가 길이 4짜리 벡터 하나입니다. y 열도 하나의 열로 다룬다는 점이 중요합니다. 모델의 예측은 마지막에 테스트 행의 y 토큰에서 읽어 냅니다(5절).

In [8]:
columns = ["무게", "당도", "y"]
grid = pd.DataFrame(
    [["[" + ", ".join(f"{v:.2f}" for v in tokens[r, c]) + "]" for c in range(3)] for r in range(6)],
    columns=columns,
    index=roles,
)
display(grid)

,무게,당도,y
학습,"[0.00, 0.46, 0.21, 0.01]","[-0.00, -0.46, -0.21, -0.01]","[0.28, 0.42, -0.48, -0.31]"
학습,"[0.01, 0.67, 0.32, 0.02]","[-0.01, -0.68, -0.32, -0.02]","[0.28, 0.42, -0.48, -0.31]"
학습,"[-0.01, -0.73, -0.34, -0.02]","[0.01, 0.68, 0.32, 0.02]","[0.22, 0.21, -1.15, -0.60]"
학습,"[-0.00, -0.40, -0.19, -0.01]","[0.00, 0.46, 0.21, 0.01]","[0.22, 0.21, -1.15, -0.60]"
테스트,"[0.01, 0.57, 0.26, 0.01]","[-0.00, -0.23, -0.11, -0.01]","[-0.13, 0.60, 0.12, -0.51]"
테스트,"[-0.01, -0.62, -0.29, -0.01]","[0.00, 0.23, 0.11, 0.01]","[-0.13, 0.60, 0.12, -0.51]"


## 3. feature attention — 같은 행의 셀끼리

한 층의 첫 단계는 같은 행 안의 토큰끼리 하는 attention입니다. 과일 표에서는 행마다 토큰 3개(무게, 당도, y)가 서로 attend합니다. 행마다 따로 계산하지만 모든 행이 같은 가중치를 씁니다.

첫 학습 행에서 attention 가중치를 직접 계산해 봅니다. head는 하나로 줄였고, 가중치 차이가 잘 보이도록 query·key 투영을 큰 값으로 뽑았습니다.

In [9]:
torch.manual_seed(1)
q_proj = nn.Linear(E, E, bias=False)  # query: "무엇을 찾는가"
k_proj = nn.Linear(E, E, bias=False)  # key: "무엇을 가지고 있는가"
nn.init.normal_(q_proj.weight, std=1.5)  # 설명용: 가중치 차이가 잘 보이도록 크게
nn.init.normal_(k_proj.weight, std=1.5)

row_tokens = tokens[0]  # 첫 학습 행의 토큰 3개: (열 3, E)
with torch.no_grad():
    q = q_proj(row_tokens)
    k = k_proj(row_tokens)
    scores = q @ k.T / E**0.5  # (3, 3): 행 = 묻는 토큰, 열 = 답하는 토큰
    weights = torch.softmax(scores, dim=-1)  # 행마다 합이 1

display(pd.DataFrame(weights.numpy(), index=columns, columns=columns).round(2))

,무게,당도,y
무게,0.45,0.28,0.27
당도,0.24,0.38,0.38
y,0.69,0.15,0.16


각 행의 합은 1입니다. 맨 아래 줄은 y 토큰이 같은 행의 무게·당도 토큰에서 정보를 얼마나 가져오는지입니다. 이 단계에서 "이 행의 feature 값과 정답"이 한 토큰에 모입니다. 테스트 행의 y 토큰은 정답이 비어 있으니, 자기 행의 feature 정보만 모읍니다.

## 4. sample attention — 같은 열의 셀끼리

둘째 단계는 같은 열 안의 토큰끼리, 즉 행 사이의 attention입니다. 열마다 따로 계산하고, 모든 열이 같은 가중치를 씁니다. 규칙은 v1의 행 attention과 같습니다.

- 학습 행은 학습 행(자기 포함)에 attend합니다.
- 테스트 행은 학습 행에만 attend합니다. 다른 테스트 행과 자기 자신은 보지 않습니다.

행 6개로 이 규칙을 표로 그리면 다음과 같습니다. 행은 attend하는 쪽, 열은 attend받는 쪽이고, 1이면 attend합니다.

In [10]:
names = [f"{role} {i + 1}" for i, role in enumerate(roles)]
col = torch.arange(len(names))
allowed = (col < n_train).expand(len(names), -1)  # 모든 행이 학습 행 열만 본다
display(pd.DataFrame(allowed.int().numpy(), index=names, columns=names))

,학습 1,학습 2,학습 3,학습 4,테스트 5,테스트 6
학습 1,1,1,1,1,0,0
학습 2,1,1,1,1,0,0
학습 3,1,1,1,1,0,0
학습 4,1,1,1,1,0,0
테스트 5,1,1,1,1,0,0
테스트 6,1,1,1,1,0,0


모든 행이 학습 행만 보므로 mask가 필요 없습니다. query는 모든 행에서, key와 value는 학습 행에서만 만들면 됩니다.

무게 열에서 가중치를 직접 계산해 봅니다. 이번에는 투영을 무작위로 뽑지 않고, 토큰을 그대로 query와 key로 씁니다(3배 키움). 그러면 attention 가중치는 두 토큰이 얼마나 닮았는지(내적)를 따릅니다. 학습한 모델은 query·key 투영을 배워서 무엇을 "닮음"으로 볼지 스스로 정합니다.

In [11]:
weight_column = tokens[:, 0]  # 무게 열의 토큰 6개: (행 6, E)
q = 3 * weight_column  # query: 모든 행
k = 3 * weight_column[:n_train]  # key: 학습 행만
weights = torch.softmax(q @ k.T / E**0.5, dim=-1)  # (6, 4)

display(pd.DataFrame(weights.numpy(), index=names, columns=names[:n_train]).round(2))

,학습 1,학습 2,학습 3,학습 4
학습 1,0.35,0.60,0.02,0.04
학습 2,0.31,0.68,0.00,0.01
학습 3,0.01,0.00,0.78,0.21
학습 4,0.05,0.03,0.62,0.30
테스트 5,0.33,0.64,0.01,0.02
테스트 6,0.01,0.01,0.74,0.25


가중치 표는 6 × 4입니다. 테스트 행이 학습 행만 보기 때문입니다. 테스트 5(200 g)는 무게가 비슷한 학습 1, 2(사과)에, 테스트 6(310 g)은 학습 3, 4(배)에 더 크게 attend합니다.

y 열에서도 같은 계산을 합니다. 테스트 행의 y 토큰은 학습 행의 y 토큰들을 읽는데, 이 토큰들에는 3절에서 모은 "그 행의 feature와 정답" 정보가 들어 있습니다. feature attention과 sample attention을 번갈아 하면, 테스트 행은 "나와 feature가 비슷한 학습 행의 정답"을 모을 수 있습니다.

## 5. 한 층 쌓기와 작은 모델

v2의 한 층은 세 단계입니다(공식 `TabPFNBlock.forward`).

1. feature attention: 행이 batch, 열이 sequence
2. sample attention: 열이 batch, 행이 sequence
3. MLP: 토큰마다 따로

각 단계 뒤에 residual 연결과 LayerNorm을 둡니다(post-norm). 위의 계산을 `nn.MultiheadAttention`으로 옮기면 다음과 같습니다. `cached_train`은 9절에서 씁니다.

In [12]:
class TwoWayLayer(nn.Module):
    # feature attention → sample attention → MLP, 각 단계 뒤에 residual + LayerNorm

    def __init__(self, emsize: int, nhead: int):
        super().__init__()
        self.feature_attention = nn.MultiheadAttention(emsize, nhead, bias=False, batch_first=True)
        self.sample_attention = nn.MultiheadAttention(emsize, nhead, bias=False, batch_first=True)
        self.mlp = nn.Sequential(
            nn.Linear(emsize, 4 * emsize, bias=False),
            nn.GELU(),
            nn.Linear(4 * emsize, emsize, bias=False),
        )
        self.norm1 = nn.LayerNorm(emsize, elementwise_affine=False)
        self.norm2 = nn.LayerNorm(emsize, elementwise_affine=False)
        self.norm3 = nn.LayerNorm(emsize, elementwise_affine=False)

    def forward(self, h, n_train, cached_train=None):
        # h: (행, 열, E)
        # 1) feature attention: 행 하나 = batch 하나, 그 행의 열 토큰들 = sequence
        out, _ = self.feature_attention(h, h, h)
        h = self.norm1(h + out)

        # 2) sample attention: 열 하나 = batch 하나, 그 열의 행 토큰들 = sequence
        h_col = h.transpose(0, 1)  # (열, 행, E)
        if cached_train is None:
            train = h_col[:, :n_train]  # key·value는 학습 행에서만
        else:
            train = cached_train  # 9절: 미리 계산해 둔 학습 행
        out, _ = self.sample_attention(h_col, train, train)
        h = self.norm2(h_col + out).transpose(0, 1)  # 다시 (행, 열, E)

        # 3) MLP: 토큰마다 따로
        h = self.norm3(h + self.mlp(h))
        return h, train

여기에 2절의 인코더와 출력층을 붙이면 작은 v2 모델이 됩니다. 출력은 테스트 행의 y 토큰(마지막 열)을 MLP에 넣어 얻습니다. 분류면 클래스 logit 10개, 회귀면 bar distribution의 구간 logit입니다. `column_embedding`은 7절에서 설명합니다.

In [13]:
class TinyTabPFNv2(nn.Module):
    def __init__(self, emsize=16, nhead=2, nlayers=2, n_out=10, column_embedding=True):
        super().__init__()
        self.emsize = emsize
        self.feature_embedder = nn.Linear(2, emsize, bias=False)  # [값, 빈 칸 표시] → 토큰
        self.target_embedder = nn.Linear(2, emsize)
        self.column_embedder = nn.Linear(emsize // 4, emsize) if column_embedding else None
        self.layers = nn.ModuleList(TwoWayLayer(emsize, nhead) for _ in range(nlayers))
        self.decoder = nn.Sequential(
            nn.Linear(emsize, 4 * emsize), nn.GELU(), nn.Linear(4 * emsize, n_out)
        )

    @staticmethod
    def fit_stats(x_train, y_train):
        # 학습 행에서만 구하는 통계: feature 평균·표준편차, y 평균
        mean = torch.nanmean(x_train, dim=0)
        filled = torch.where(torch.isnan(x_train), mean, x_train)
        std = filled.std(dim=0, correction=0).clamp_min(1e-6)
        return mean, std, y_train.mean()

    def column_embeddings(self, n_columns):
        # 7절: 열마다 고정된 무작위 벡터를 학습하는 Linear에 통과시킨다
        generator = torch.Generator().manual_seed(0)  # 항상 같은 무작위 벡터
        random_vectors = torch.randn(n_columns, self.emsize // 4, generator=generator)
        return self.column_embedder(random_vectors)  # (열, E)

    def embed_features(self, x, mean, std):
        missing = torch.isnan(x)
        z = (torch.where(missing, mean, x) - mean) / std  # 빈 칸은 평균으로 채운 뒤 표준화
        indicator = missing.float() * NAN_INDICATOR
        h = self.feature_embedder(torch.stack([z, indicator], dim=-1))  # (행, 열, E)
        if self.column_embedder is not None:
            h = h + self.column_embeddings(x.shape[1])  # 같은 열의 모든 행에 같은 벡터
        return h

    def embed_target(self, y, y_mean):
        missing = torch.isnan(y)
        filled = torch.where(missing, y_mean, y)
        indicator = missing.float() * NAN_INDICATOR
        return self.target_embedder(torch.stack([filled, indicator], dim=-1)).unsqueeze(1)

    def forward(self, x_train, y_train, x_test):
        n_train = len(x_train)
        mean, std, y_mean = self.fit_stats(x_train, y_train)
        x = torch.cat([x_train, x_test])
        y = torch.cat([y_train, torch.full((len(x_test),), float("nan"))])  # 테스트 행 y는 빈 칸
        h = torch.cat([self.embed_features(x, mean, std), self.embed_target(y, y_mean)], dim=1)
        for layer in self.layers:
            h, _ = layer(h, n_train)
        return self.decoder(h[n_train:, -1])  # 테스트 행의 y 토큰 → logit

과일 표를 넣어 shape을 따라가 봅니다. 학습하지 않은 모델이라 확률 값 자체에는 의미가 없습니다.

In [14]:
x_train = torch.tensor(features[:n_train], dtype=torch.float32)
y_train = torch.tensor(y[:n_train], dtype=torch.float32)
x_test = torch.tensor(features[n_train:], dtype=torch.float32)

torch.manual_seed(0)
tiny = TinyTabPFNv2().eval()
with torch.no_grad():
    logits = tiny(x_train, y_train, x_test)
proba = torch.softmax(logits[:, :2], dim=-1)  # 클래스가 2개이므로 앞의 2개만

print("x_train:", tuple(x_train.shape), " (학습 행, feature)")
print("x_test: ", tuple(x_test.shape), " (테스트 행, feature)")
print("logits: ", tuple(logits.shape), " (테스트 행, 출력 10개)")
display(pd.DataFrame(proba.numpy(), columns=["P(사과)", "P(배)"], index=names[n_train:]).round(3))

x_train: (4, 2)  (학습 행, feature)
x_test:  (2, 2)  (테스트 행, feature)
logits:  (2, 10)  (테스트 행, 출력 10개)


,P(사과),P(배)
테스트 5,0.548,0.452
테스트 6,0.549,0.451


## 6. 구조가 보장하는 성질

이 구조에서는 학습하지 않은 모델에서도 네 가지 성질이 성립합니다.

1. **학습 행 순서**: 학습 행의 순서를 바꿔도 예측이 같습니다. sample attention에 위치 정보가 없기 때문입니다(v1과 같음).
2. **테스트 행 독립**: 테스트 행 하나의 예측은 다른 테스트 행과 무관합니다(v1과 같음).
3. **feature 열 순서**: feature attention에도 위치 정보가 없으므로, 열 순서를 바꿔도 예측이 같습니다. 단, 7절의 column embedding을 끄면 그렇습니다.
4. **feature 수**: 같은 모델에 feature 수가 다른 표를 그대로 넣을 수 있습니다. 0으로 채울 필요가 없습니다.

feature 5개, 학습 행 20개, 테스트 행 6개짜리 무작위 표로 확인합니다. 비교용으로 학습 행 하나의 정답을 바꾼 경우도 봅니다. 이때는 예측이 달라져야 합니다.

In [15]:
torch.manual_seed(0)
x_tr = torch.randn(20, 5)
y_tr = torch.randint(0, 3, (20,)).float()
x_te = torch.randn(6, 5)

torch.manual_seed(0)
plain = TinyTabPFNv2(column_embedding=False).eval()  # column embedding 없음
torch.manual_seed(0)
tagged = TinyTabPFNv2(column_embedding=True).eval()  # column embedding 있음


def max_diff(a, b):
    return f"{(a - b).abs().max().item():.1e}"


with torch.no_grad():
    base = plain(x_tr, y_tr, x_te)

    row_perm = torch.randperm(20)
    rows_shuffled = plain(x_tr[row_perm], y_tr[row_perm], x_te)

    one_by_one = torch.cat([plain(x_tr, y_tr, x_te[j : j + 1]) for j in range(6)])

    col_perm = torch.tensor([3, 0, 4, 1, 2])
    cols_shuffled = plain(x_tr[:, col_perm], y_tr, x_te[:, col_perm])

    tagged_base = tagged(x_tr, y_tr, x_te)
    tagged_shuffled = tagged(x_tr[:, col_perm], y_tr, x_te[:, col_perm])

    y_changed = y_tr.clone()
    y_changed[0] = (y_changed[0] + 1) % 3
    label_changed = plain(x_tr, y_changed, x_te)

display(
    pd.DataFrame(
        {
            "바꾼 것": [
                "1. 학습 행 순서",
                "2. 테스트 행을 하나씩 따로 넣기",
                "3. feature 열 순서 (column embedding 없음)",
                "3. feature 열 순서 (column embedding 있음)",
                "비교: 학습 행 하나의 정답",
            ],
            "logit 최대 차이": [
                max_diff(base, rows_shuffled),
                max_diff(base, one_by_one),
                max_diff(base, cols_shuffled),
                max_diff(tagged_base, tagged_shuffled),
                max_diff(base, label_changed),
            ],
        }
    )
)

,바꾼 것,logit 최대 차이
0,1. 학습 행 순서,1.2e-07
1,2. 테스트 행을 하나씩 따로 넣기,1.2e-07
2,3. feature 열 순서 (column embedding 없음),2.4e-07
3,3. feature 열 순서 (column embedding 있음),7.9e-02
4,비교: 학습 행 하나의 정답,5.3e-03


1, 2와 column embedding이 없는 3은 차이가 float 반올림 오차 수준입니다. column embedding이 있으면 열 순서에 따라 예측이 달라지고, 정답을 바꿔도 달라집니다. 학습하지 않은 모델이라 차이의 크기 자체에는 의미가 없고, 0인지 아닌지만 보면 됩니다.

4번은 같은 모델에 feature 3개짜리 표와 8개짜리 표를 그대로 넣어 보면 됩니다.

In [16]:
with torch.no_grad():
    for n_features in [3, 8]:
        out = plain(torch.randn(20, n_features), y_tr, torch.randn(6, n_features))
        print(f"feature {n_features}개 → logits {tuple(out.shape)}")

feature 3개 → logits (6, 10)
feature 8개 → logits (6, 10)


v1은 `x_encoder`의 입력 크기가 feature 상한(100)을 정했지만, v2에는 그런 층이 없습니다. 논문은 feature 500개까지 평가했고, 학습 때는 feature를 160개까지만 봤습니다(Methods "Training details"). 학습 때보다 큰 표에서도 동작하는 이유가 이 구조입니다.

## 7. 열에 이름표 붙이기 — random feature embedding

6절의 3번 성질에는 대가가 있습니다. 토큰에는 "몇 번째 열"이라는 정보가 없고, 모델은 값으로만 열을 구분합니다. 논문은 같은 값들을 순서만 바꿔 담은 두 feature처럼 통계가 같은 열을 모델이 구분하기 어렵다고 적었습니다(Methods). 극단적인 예로, 무게 열을 그대로 복사한 열을 붙여 봅니다.

In [17]:
dup = torch.cat([x_train[:, :1], x_train[:, :1], x_train[:, 1:]], dim=1)  # 무게, 무게(복사), 당도
dup_test = torch.cat([x_test[:, :1], x_test[:, :1], x_test[:, 1:]], dim=1)


def last_layer_tokens(model, x_tr, y_tr, x_te):
    # 마지막 층을 지난 토큰 (행, 열, E)
    mean, std, y_mean = model.fit_stats(x_tr, y_tr)
    x = torch.cat([x_tr, x_te])
    y_all = torch.cat([y_tr, torch.full((len(x_te),), float("nan"))])
    h = torch.cat([model.embed_features(x, mean, std), model.embed_target(y_all, y_mean)], dim=1)
    for layer in model.layers:
        h, _ = layer(h, len(x_tr))
    return h


with torch.no_grad():
    for name, model in [("column embedding 없음", plain), ("column embedding 있음", tagged)]:
        h = last_layer_tokens(model, dup, y_train, dup_test)
        gap = (h[:, 0] - h[:, 1]).abs().max().item()
        print(f"{name}: 무게 열과 복사한 열의 토큰 차이 = {gap:.1e}")

column embedding 없음: 무게 열과 복사한 열의 토큰 차이 = 0.0e+00
column embedding 있음: 무게 열과 복사한 열의 토큰 차이 = 2.8e+00


column embedding이 없으면 두 열의 토큰은 처음부터 끝까지 똑같습니다. 입력이 같고 모든 열이 같은 가중치를 쓰니, 몇 층을 지나도 갈라질 수 없습니다. 실제 표에서는 두 열이 완전히 같은 경우는 드물지만, 값의 분포가 비슷한 열은 흔합니다.

v2는 열마다 무작위 벡터 하나를 뽑고, 그 벡터를 학습하는 Linear에 통과시켜 그 열의 모든 토큰에 더합니다. 무작위 벡터의 크기는 임베딩의 1/4이고(공식 v2는 192/4 = 48), 시드를 고정해 항상 같은 벡터를 씁니다. 첫 열에는 항상 같은 이름표가, 둘째 열에는 다른 이름표가 붙는 셈입니다.

대신 6절에서 본 것처럼 열 순서에 따라 예측이 달라집니다. 그래서 v2도 추론할 때 feature 순서를 섞은 앙상블을 씁니다(논문 "Inference details"). 열 순서에 완전히 무관한 모델과 열을 확실히 구분하는 모델 사이에서, v2는 구분하는 쪽을 고르고 앙상블로 순서 의존을 줄입니다.

## 8. 셀 2개를 토큰 하나로 — feature group

논문은 셀 하나 대신 셀 여러 개를 묶어 토큰 하나로 만드는 편이 더 낫다고 적었습니다(Methods). 공식 v2 기본 모델은 feature 2개를 묶습니다(`features_per_group = 2`). 토큰 하나의 입력은 [값 1, 값 2, 빈 칸 표시 1, 빈 칸 표시 2] 네 숫자가 되고, Linear(4 → E)로 토큰을 만듭니다.

feature가 홀수 개면 마지막 묶음의 빈자리를 0으로 채웁니다. 그러면 그 묶음은 값이 하나뿐이라 다른 묶음보다 크기가 작아집니다. 공식 구현은 묶음마다 $\sqrt{2 / \text{실제로 쓴 feature 수}}$를 곱해 크기를 맞춥니다. v1이 0으로 채운 뒤 $100/k$배 했던 것(v1 walkthrough 2.4절)을 묶음 단위로 한 것입니다. feature 3개짜리 표로 따라가 봅니다.

In [18]:
small = pd.DataFrame({"f1": [0.5, -1.2, 0.7], "f2": [1.0, 0.3, -1.3], "f3": [-0.4, 0.9, -0.5]})

# 1단계: 열 수를 2의 배수로 맞춘다 (빈자리는 0)
padded = small.copy()
padded["f4 (빈자리)"] = 0.0

# 2단계: 2개씩 묶고, 묶음마다 sqrt(2 / 실제로 쓴 feature 수)를 곱한다
group_scale = {"묶음 1 (f1, f2)": np.sqrt(2 / 2), "묶음 2 (f3, 빈자리)": np.sqrt(2 / 1)}
grouped = pd.DataFrame(
    {
        "묶음 1 (f1, f2)": [
            f"[{a * group_scale['묶음 1 (f1, f2)']:.2f}, {b * group_scale['묶음 1 (f1, f2)']:.2f}]"
            for a, b in zip(padded["f1"], padded["f2"], strict=True)
        ],
        "묶음 2 (f3, 빈자리)": [
            f"[{a * group_scale['묶음 2 (f3, 빈자리)']:.2f}, {b:.2f}]"
            for a, b in zip(padded["f3"], padded["f4 (빈자리)"], strict=True)
        ],
    }
)
print("원래 표 (feature 3개)")
display(small)
print("묶은 뒤 (토큰 2개, 묶음 2는 √2배)")
display(grouped)

원래 표 (feature 3개)


,f1,f2,f3
0,0.5,1.0,-0.4
1,-1.2,0.3,0.9
2,0.7,-1.3,-0.5


묶은 뒤 (토큰 2개, 묶음 2는 √2배)


,"묶음 1 (f1, f2)","묶음 2 (f3, 빈자리)"
0,"[0.50, 1.00]","[-0.57, 0.00]"
1,"[-1.20, 0.30]","[1.27, 0.00]"
2,"[0.70, -1.30]","[-0.71, 0.00]"


묶으면 행마다 feature 토큰 수가 절반이 됩니다. feature attention의 비용은 토큰 수의 제곱이므로 약 1/4로, sample attention은 열 수에 비례하므로 약 1/2로 줄어듭니다. 논문의 하이퍼파라미터 탐색에서 고른 분류 모델 6개 중 3개, 회귀 모델 5개 중 4개가 2개씩 묶는 구조였고, 기본 모델도 여기에 들어갑니다.

## 9. 학습 행은 한 번만 계산하기 — 상태 캐시

4절의 규칙 때문에 학습 행의 표현은 테스트 행의 영향을 받지 않습니다. 그래서 학습 행만 먼저 넣어 층마다 학습 행의 상태를 저장해 두고, 나중에 테스트 행만 넣어 그 상태에 attend하게 할 수 있습니다. 보통의 머신러닝 모델처럼 `fit` 한 번, `predict` 여러 번이 됩니다. 논문은 학습 행 10,000개, feature 10개에서 이렇게 하면 CPU 추론이 32초에서 0.1초로 빨라진다고 적었습니다.

작은 모델에 `fit`과 `predict`를 붙여, 한 번에 넣은 결과와 같은지 확인합니다.

In [19]:
def fit(model, x_train, y_train):
    # 학습 행만 넣고, 층마다 sample attention의 key·value가 될 학습 행 상태를 저장한다
    stats = model.fit_stats(x_train, y_train)
    mean, std, y_mean = stats
    h = torch.cat(
        [model.embed_features(x_train, mean, std), model.embed_target(y_train, y_mean)], dim=1
    )
    states = []
    for layer in model.layers:
        h, train_state = layer(h, len(x_train))
        states.append(train_state)
    return stats, states


def predict(model, cache, x_test):
    # 테스트 행만 넣고, 저장해 둔 학습 행 상태에 attend한다
    (mean, std, y_mean), states = cache
    y_empty = torch.full((len(x_test),), float("nan"))
    h = torch.cat(
        [model.embed_features(x_test, mean, std), model.embed_target(y_empty, y_mean)], dim=1
    )
    for layer, train_state in zip(model.layers, states, strict=True):
        h, _ = layer(h, 0, cached_train=train_state)
    return model.decoder(h[:, -1])


with torch.no_grad():
    cache = fit(tagged, x_tr, y_tr)  # 학습 행은 한 번만
    first_half = predict(tagged, cache, x_te[:3])  # 테스트 행은 나눠서 여러 번
    second_half = predict(tagged, cache, x_te[3:])
    joint = tagged(x_tr, y_tr, x_te)  # 한 번에 넣은 결과

print("한 번에 넣은 결과와의 최대 차이:", max_diff(joint, torch.cat([first_half, second_half])))

한 번에 넣은 결과와의 최대 차이: 1.0e-07


결과가 같습니다. 공식 구현은 여기서 한 가지를 더 줄입니다. 테스트 행의 sample attention은 head가 6개여도 key·value는 첫 head 하나만 씁니다(multi-query attention). 그래서 캐시에 학습 행 셀마다 key·value 한 쌍만 저장하면 됩니다(`AlongColumnAttention.forward`). 학습 행끼리는 head 6개를 모두 씁니다.

## 10. 계산량

토큰이 셀 단위가 되면서 데이터셋 하나를 처리하는 계산은 v1보다 커집니다. 층 하나에서 계산하는 attention 점수의 개수를 비교합니다. 학습 행 $n$, 테스트 행 $m$, feature $k$개이고, v2는 2개씩 묶어 행마다 토큰이 $g = \lceil k/2 \rceil + 1$개(y 토큰 포함)입니다.

- v1: 행 사이 attention 한 번, $n^2 + n m$
- v2 feature attention: 행마다 $g^2$, 모두 $(n + m)\, g^2$
- v2 sample attention: 열마다 $n^2 + n m$, 모두 $g\,(n^2 + n m)$

In [20]:
rows = []
for n, m, k in [(1_000, 1_000, 100), (10_000, 1_000, 10), (10_000, 1_000, 500)]:
    g = -(-k // 2) + 1  # 묶은 feature 토큰 + y 토큰
    v1 = n**2 + n * m
    feature_attn = (n + m) * g**2
    sample_attn = g * (n**2 + n * m)
    rows.append(
        {
            "학습 행 n": f"{n:,}",
            "테스트 행 m": f"{m:,}",
            "feature k": k,
            "v1": f"{v1:,}",
            "v2 feature attn": f"{feature_attn:,}",
            "v2 sample attn": f"{sample_attn:,}",
            "v2 / v1": f"{(feature_attn + sample_attn) / v1:.0f}배",
        }
    )
display(pd.DataFrame(rows))

,학습 행 n,테스트 행 m,feature k,v1,v2 feature attn,v2 sample attn,v2 / v1
0,"1,000","1,000",100,"2,000,000","5,202,000","102,000,000",54배
1,"10,000","1,000",10,"110,000,000","396,000","660,000,000",6배
2,"10,000","1,000",500,"110,000,000","693,011,000","27,610,000,000",257배


v2는 대부분의 계산이 sample attention에서 나오고, 열(묶음) 수만큼 v1보다 많습니다. 행 수에는 v1과 같이 제곱으로 늘고, feature 수에는 대략 비례해서 늘어납니다. 논문은 이를 $O(n^2 + m^2)$로 적었습니다(논문의 $n$은 행 수, $m$은 feature 수).

그래도 v2가 v1보다 큰 표를 다룰 수 있는 이유는 메모리입니다. 행과 열을 따로 attend하므로 큰 attention 행렬을 한 번에 만들 필요가 없고, 논문은 flash attention, half-precision LayerNorm, activation checkpointing을 더해 셀 하나당 1,000 byte 미만으로 줄였다고 적었습니다. H100 한 장에서 셀 5,000만 개(예: 500만 행 × 10열)까지 예측할 수 있다고 합니다.

In [21]:
bytes_per_cell = 1_000  # 논문: 셀 하나당 1,000 byte 미만
for n_rows, n_cols in [(10_000, 500), (5_000_000, 10)]:
    cells = n_rows * n_cols
    gigabytes = cells * bytes_per_cell / 1e9
    print(f"{n_rows:>9,}행 × {n_cols:>3}열 = 셀 {cells:>11,}개 → 약 {gigabytes:.0f} GB 이하")

   10,000행 × 500열 = 셀   5,000,000개 → 약 5 GB 이하
5,000,000행 ×  10열 = 셀  50,000,000개 → 약 50 GB 이하


H100의 메모리는 80 GB이므로 셀 5,000만 개(50 GB)가 들어갑니다. 논문이 평가한 가장 큰 표(10,000행 × 500열)는 5 GB 정도입니다.

## 11. 정리

| | v1 | v2 | 이 노트북 |
|---|---|---|---|
| 토큰 | 행 하나 | 셀 하나 (기본 모델은 셀 2개) | 2절, 8절 |
| 결측값 | 테스트 때 0으로 바꿈 | 평균으로 채우고 빈 칸 표시($-2$)를 함께 넣음 | 2절 |
| attention | 행 사이 한 번 | feature attention → sample attention 번갈아 | 3~5절 |
| 테스트 행 | 학습 행에만 attend | 같음 (열마다) | 4절 |
| 예측을 읽는 곳 | 테스트 행 토큰 | 테스트 행의 y 토큰 | 5절 |
| 열 순서 | `x_encoder` 가중치가 위치마다 다름 | 구조는 무관, column embedding으로 이름표를 붙임 | 6절, 7절 |
| feature 수 | 100 (0으로 채움) | 구조상 제한 없음 (학습 160, 평가 500) | 6절 |
| 학습 행 상태 | 매번 다시 계산 | 한 번 계산해 캐시 | 9절 |

다음 노트북 [`02_prior_primer`](02_prior_primer.ipynb)에서는 v2가 학습하는 합성 데이터셋, 즉 새 prior를 봅니다.